# Driver Cellphone Use YOLO Training Template

This notebook converts the attached Roboflow COCO dataset to YOLO format, validates it, fine-tunes YOLO on Kaggle GPU, evaluates the trained model, benchmarks inference, and zips useful artifacts for download.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/tinhvu11235/homeobjects-yolo-detector.git'
PROJECT_DIR = Path('/kaggle/working/homeobjects-yolo-detector')
SRC_DIR = PROJECT_DIR / 'src'
SCRIPTS_DIR = PROJECT_DIR / 'scripts'
DATASET_CHECK = SRC_DIR / 'dataset_check.py'
TRAIN_SCRIPT = SRC_DIR / 'train.py'
EVALUATE_SCRIPT = SRC_DIR / 'evaluate.py'
BENCHMARK_SCRIPT = SRC_DIR / 'benchmark.py'
CONVERTER_SCRIPT = SCRIPTS_DIR / 'convert_coco_to_yolo.py'

if PROJECT_DIR.exists():
    print(f'Repository already exists: {PROJECT_DIR}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())
print('Python:', sys.version)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

import torch
import ultralytics

print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import yaml

INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    raise FileNotFoundError('/kaggle/input does not exist. Attach the Roboflow COCO dataset first.')

print('Attached Kaggle datasets:')
for child in sorted(INPUT_ROOT.iterdir()):
    print('-', child)

annotation_paths = sorted(INPUT_ROOT.rglob('_annotations.coco.json'))
if not annotation_paths:
    raise FileNotFoundError('No _annotations.coco.json file found. Attach the COCO export, not only images.')

candidate_roots = sorted({path.parent.parent for path in annotation_paths})
def root_score(path):
    text = str(path).lower()
    score = 0
    for keyword in ['cellphone', 'phone', 'driver']:
        if keyword in text:
            score += 10
    if (path / 'train' / '_annotations.coco.json').exists():
        score += 5
    if (path / 'valid' / '_annotations.coco.json').exists():
        score += 5
    return score

SOURCE_COCO_ROOT = max(candidate_roots, key=root_score)
CONVERTED_DATASET = Path('/kaggle/working/cellphone_drivers_yolo')
print('Using COCO root:', SOURCE_COCO_ROOT)
print('Converted YOLO dataset:', CONVERTED_DATASET)

subprocess.run([
    sys.executable,
    str(CONVERTER_SCRIPT),
    '--source', str(SOURCE_COCO_ROOT),
    '--output', str(CONVERTED_DATASET),
], check=True)

DATA_YAML = CONVERTED_DATASET / 'data.yaml'
with DATA_YAML.open('r', encoding='utf-8') as file:
    DATA_CONFIG = yaml.safe_load(file) or {}
print('Using dataset YAML:', DATA_YAML)
print('YAML keys:', sorted(DATA_CONFIG.keys()))
print('Classes:', DATA_CONFIG.get('names'))

In [ ]:
!python "{DATASET_CHECK}" --data "{DATA_YAML}" --output "{PROJECT_DIR / 'outputs' / 'dataset_report_cellphone_drivers.json'}"

In [ ]:
!python "{TRAIN_SCRIPT}" \
  --data "{DATA_YAML}" \
  --model yolo11s.pt \
  --epochs 80 \
  --imgsz 640 \
  --batch 16 \
  --device 0 \
  --project "{PROJECT_DIR / 'runs'}" \
  --name cellphone_drivers_yolo

In [ ]:
BEST_PT = PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'weights' / 'best.pt'
print('best.pt:', BEST_PT, 'exists:', BEST_PT.exists())
if not BEST_PT.exists():
    raise FileNotFoundError('Training did not produce best.pt.')

# Required post-training evaluation on validation split.
!python "{EVALUATE_SCRIPT}" --weights "{BEST_PT}" --data "{DATA_YAML}" --imgsz 640 --device 0 --split val --output "{PROJECT_DIR / 'outputs' / 'eval_summary_val.json'}"

# Additional held-out test evaluation.
!python "{EVALUATE_SCRIPT}" --weights "{BEST_PT}" --data "{DATA_YAML}" --imgsz 640 --device 0 --split test --output "{PROJECT_DIR / 'outputs' / 'eval_summary_test.json'}"

In [ ]:
VAL_IMAGES = CONVERTED_DATASET / 'images' / 'val'
if not VAL_IMAGES.exists():
    raise FileNotFoundError(f'Validation image folder not found: {VAL_IMAGES}')

!python "{BENCHMARK_SCRIPT}" \
  --weights "{BEST_PT}" \
  --source "{VAL_IMAGES}" \
  --imgsz 640 \
  --device 0 \
  --warmup 20 \
  --runs 500 \
  --save-json "{PROJECT_DIR / 'outputs' / 'benchmark_summary.json'}" \
  --save-csv "{PROJECT_DIR / 'outputs' / 'benchmark_results.csv'}"

In [ ]:
import zipfile

artifact_paths = [
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'weights' / 'best.pt',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'weights' / 'last.pt',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'results.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'confusion_matrix.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'PR_curve.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'F1_curve.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'args.yaml',
    PROJECT_DIR / 'outputs' / 'dataset_report_cellphone_drivers.json',
    PROJECT_DIR / 'outputs' / 'eval_summary_val.json',
    PROJECT_DIR / 'outputs' / 'eval_summary_test.json',
    PROJECT_DIR / 'outputs' / 'benchmark_summary.json',
    PROJECT_DIR / 'outputs' / 'benchmark_results.csv',
]

zip_path = PROJECT_DIR / 'cellphone_drivers_yolo_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
    for path in artifact_paths:
        if path.exists():
            zip_file.write(path, arcname=str(path.relative_to(PROJECT_DIR)))
        else:
            print('Missing artifact, skipped:', path)

print('Created:', zip_path.resolve())
print('Download best.pt and place it in models/best.pt for Hugging Face deployment.')

## Deployment Step

After training and evaluation, download `runs/cellphone_drivers_yolo/weights/best.pt` from Kaggle output and place it into `models/best.pt` in the repository. Hugging Face Spaces will run `app.py` for CPU inference only.